### Pregunta 6
“Hermandades ocultas entre países”
Organiza países por similitud y muestra un heatmap clusterizado. Cuenta qué dos países resultan “hermanos” inesperados.

In [2]:
df= pd.read_csv("Datos-1.csv")

In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage
import plotly.figure_factory as ff

In [4]:
r6 = df.copy()
r6.drop(['Happiness Rank', 'Happiness Score', 'Region', 'Standard Error'], axis=1, inplace=True)

# Usamos el país como índice
r6.set_index('Country', inplace=True)
r6 = r6.select_dtypes(include=['float64', 'int64'])

# Normalizamos todas las variables restantes
scaler = StandardScaler()
df_normalizado = pd.DataFrame(
    scaler.fit_transform(r6),
    columns=r6.columns,
    index=r6.index
)

# --- Clustering jerárquico ---
data_matrix = df_normalizado.values
linked_matrix = linkage(data_matrix, method='average', metric='euclidean')

# --- Dendrograma con heatmap clusterizado ---
fig = ff.create_dendrogram(
    df_normalizado,
    orientation='right',
    labels=df_normalizado.index.tolist(),
)

# Reordenamos el heatmap según el dendrograma
for i in range(len(fig['data'])):
    if fig['data'][i]['type'] == 'heatmap':
        reordered_countries = fig['layout']['yaxis']['ticktext']

        fig['data'][i].update(
            z=df_normalizado.loc[reordered_countries, :].values,
            x=df_normalizado.columns.tolist(),
            y=reordered_countries,
            colorscale='RdBu',
            colorbar=dict(title='Valor Normalizado (Z-Score)'),
            hovertemplate='Factor: %{x}<br>País: %{y}<br>Z-Score: %{z}<extra></extra>'
        )

# --- Ajustes de layout ---
fig.update_layout(
    title={
        'text': 'Hermandades Ocultas: Mapa de Calor Clusterizado de Perfiles de Felicidad',
        'y':0.95,
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': {'size': 20}
    },
    width=1200,
    height=1000,
    margin={'r': 200},
    xaxis_title='Factores de Felicidad',
    yaxis=dict(tickangle=0, tickfont={'size': 8})
)

fig.show()

Tras normalizar los factores de felicidad y aplicar un análisis de clustering jerárquico, se observa que los países tienden a agruparse en bloques regionales esperados, como los nórdicos entre sí, pero también aparecen emparejamientos sorprendentes: un caso llamativo es el de Costa Rica y Nueva Zelanda, que a pesar de pertenecer a regiones geográficas y contextos económicos muy distintos, muestran perfiles de bienestar muy similares en variables como apoyo social, salud y libertad percibida, lo que los convierte en “hermanos ocultos” dentro del mapa de calor clusterizado.